# Step 3 — Game Name Index & Lookup

## Goal

Build a fast and typo-tolerant system for identifying games from user queries.

The lookup system will support:

1. Normalized game names
2. Exact matching
3. Substring matching
4. Fuzzy matching
5. Top-3 game suggestions for misspelled or ambiguous queries

This component will later be used by the entity extractor, information engine, search engine, and chatbot.

In [1]:
import pandas as pd

DATA_PATH = "../data/processed/games_clean.parquet"

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)

Shape: (89550, 53)


In [2]:
print(df["name"].head(10))
print("\nMissing names:", df["name"].isna().sum())
print("Unique names:", df["name"].nunique())

0                   Counter-Strike 2
1                PUBG: BATTLEGROUNDS
2                             Dota 2
3          Grand Theft Auto V Legacy
4    Tom Clancy's Rainbow Six® Siege
5    Tom Clancy's Rainbow Six® Siege
6    Tom Clancy's Rainbow Six® Siege
7    Tom Clancy's Rainbow Six® Siege
8    Tom Clancy's Rainbow Six® Siege
9                    Team Fortress 2
Name: name, dtype: object

Missing names: 0
Unique names: 88805


## 3.1 Normalize Game Names

Create a `name_clean` column that:

- Converts names to lowercase
- Removes punctuation
- Replaces multiple spaces with a single space
- Removes leading and trailing whitespace

This normalized form will be used for exact, substring, and fuzzy matching.

In [19]:
df["name_clean"] = (
    df["name"]
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace("_", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [18]:
print(df[["name", "name_clean"]].head(20))

                               name                      name_clean
0                  Counter-Strike 2                counter strike 2
1               PUBG: BATTLEGROUNDS              pubg battlegrounds
2                            Dota 2                          dota 2
3         Grand Theft Auto V Legacy       grand theft auto v legacy
4   Tom Clancy's Rainbow Six® Siege  tom clancy s rainbow six siege
5   Tom Clancy's Rainbow Six® Siege  tom clancy s rainbow six siege
6   Tom Clancy's Rainbow Six® Siege  tom clancy s rainbow six siege
7   Tom Clancy's Rainbow Six® Siege  tom clancy s rainbow six siege
8   Tom Clancy's Rainbow Six® Siege  tom clancy s rainbow six siege
9                   Team Fortress 2                 team fortress 2
10                         Terraria                        terraria
11                             Rust                            rust
12                      Garry's Mod                     garry s mod
13                    Apex Legends™             

In [17]:
print("Missing name_clean:", df["name_clean"].isna().sum())
print("Unique name_clean:", df["name_clean"].nunique())
print("Empty name_clean:", (df["name_clean"] == "").sum())

Missing name_clean: 0
Unique name_clean: 88442
Empty name_clean: 0


In [14]:
empty_names = df.loc[df["name_clean"] == "", ["appid", "name"]]

print(empty_names.to_string(index=False))

Empty DataFrame
Columns: [appid, name]
Index: []


## 3.1.1 Remove Unusable Game Names

After Unicode-aware name normalization, four rows contained names consisting only of symbols or punctuation and therefore produced an empty `name_clean`.

These rows cannot be reliably identified through the game-name lookup system, so they are removed.

In [21]:
before = len(df)

df = df[df["name_clean"] != ""].copy()

after = len(df)

print("Rows removed:", before - after)
print("New shape:", df.shape)

Rows removed: 0
New shape: (89546, 54)


In [20]:
print("Empty name_clean:", (df["name_clean"] == "").sum())
print("Missing name_clean:", df["name_clean"].isna().sum())

Empty name_clean: 0
Missing name_clean: 0


## 3.2 Build Game Name Index

Create a dictionary that maps each normalized game name (`name_clean`)
to the corresponding row index in the DataFrame.

When multiple games have the same normalized name, keep the entry
with the highest `num_reviews_total`, so the most popular version
is selected.

The resulting index will be used for fast game-name lookup.

In [22]:
# Sort games by total reviews, highest first
order = df.sort_values(
    "num_reviews_total",
    ascending=False,
    na_position="last"
)

# Build name → DataFrame index mapping
name_index = {}

for idx, name in zip(order.index, order["name_clean"]):
    name_index.setdefault(name, idx)

# List of normalized names for fuzzy matching later
names_list = list(name_index.keys())

print("Total games:", len(df))
print("Unique names:", len(names_list))
print("Index entries:", len(name_index))

Total games: 89546
Unique names: 88442
Index entries: 88442


In [23]:
print("Duplicate normalized names:",
      df["name_clean"].duplicated().sum())

print("Name index size:", len(name_index))

Duplicate normalized names: 1104
Name index size: 88442


## 3.3 Matching Layers

GameGuide will use three matching strategies:

1. Exact match — fastest and most reliable.
2. Substring match — handles queries containing the game name.
3. Fuzzy match — handles spelling mistakes and small variations.

RapidFuzz will be used for fuzzy matching.

In [24]:
from rapidfuzz import process, fuzz

print("RapidFuzz imported successfully!")

RapidFuzz imported successfully!


### 3.3.1 Game Lookup Function

The `find_game()` function identifies a game using three matching layers:

1. Exact match against `name_clean`
2. Substring match when the query contains a game name
3. Fuzzy match using RapidFuzz for misspellings

The function returns the matched DataFrame index and a matching score.

In [53]:
import re

In [54]:
def find_game(text, cutoff=92):
    # Normalize the user's query
    t = (
        str(text)
        .lower()
        .strip()
        .replace("-", " ")
    )
    
    # Collapse extra spaces
    t = " ".join(t.split())

    # Layer 1: Exact match
    if t in name_index:
        return name_index[t], 100, "exact"

    # Layer 2: Substring match using whole words
    query_words = set(t.split())

    substring_matches = []

    for name in names_list:
        if len(name) < 3:
            continue

        # Ignore very short single-word game names
        if len(name.split()) == 1 and len(name) < 4:
            continue

        #Check whether the query appears as a complete phrase in the game name
        pattern = rf"(?<!\w){re.escape(t)}(?!\w)"

        if re.search(pattern, name):
            substring_matches.append(name)

        name_words = set(name.split())

        # Game name appears as a complete word sequence in the query
        if name_words.issubset(query_words):
            substring_matches.append(name)

    if substring_matches:
        # Prefer the longest / most specific game name
        best_name = max(
            substring_matches,
            key=lambda x: (len(x.split()), len(x))
        )
        return name_index[best_name], 95, "substring"

    # Layer 3: Fuzzy match
    match = process.extractOne(
        t,
        names_list,
        scorer=fuzz.WRatio,
        score_cutoff=cutoff
    )

    if match:
        matched_name, score, _ = match
        return name_index[matched_name], score, "fuzzy"

    # No match
    return None, 0, "none"

In [55]:
idx, score, method = find_game("Elden Ring")

print("Index:", idx)
print("Score:", score)
print("Method:", method)

if idx is not None:
    print("Game:", df.loc[idx, "name"])

Index: 16
Score: 100
Method: exact
Game: ELDEN RING


In [56]:
idx, score, method = find_game("tell me about Elden Ring")

print("Index:", idx)
print("Score:", score)
print("Method:", method)

if idx is not None:
    print("Game:", df.loc[idx, "name"])

Index: 16
Score: 95
Method: substring
Game: ELDEN RING


In [57]:
idx, score, method = find_game("eldn ring")

print("Index:", idx)
print("Score:", score)
print("Method:", method)

if idx is not None:
    print("Game:", df.loc[idx, "name"])

Index: 16
Score: 94.73684210526316
Method: fuzzy
Game: ELDEN RING


## 3.4 Fuzzy Game Suggestions

Create a function that returns the top 3 closest game-name matches for a user query.

This will be used when the user's input is misspelled or does not produce
a sufficiently confident exact/substring match.

The suggestions will be ranked using RapidFuzz's WRatio scorer.

In [58]:
def suggest(text, n=3):
    query = (
        str(text)
        .lower()
        .strip()
    )
    
    query = " ".join(query.split())

    matches = process.extract(
        query,
        names_list,
        scorer=fuzz.WRatio,
        limit=n
    )

    return [match[0] for match in matches]

In [59]:
suggestions = suggest("eldn ring")

print("Suggestions:")
for i, game in enumerate(suggestions, start=1):
    print(f"{i}. {game}")

Suggestions:
1. elden ring
2. ring runner flight of the sages
3. ring of titans


In [60]:
idx, score, method = find_game("eldn ring")

print("Test 1")
print("Game:", df.loc[idx, "name"] if idx is not None else None)
print("Score:", score)
print("Method:", method)

Test 1
Game: ELDEN RING
Score: 94.73684210526316
Method: fuzzy


In [61]:
idx, score, method = find_game("witcher 3")

print("Test 2")
print("Game:", df.loc[idx, "name"] if idx is not None else None)
print("Score:", score)
print("Method:", method)

Test 2
Game: The Witcher 3: Wild Hunt
Score: 95
Method: substring


In [62]:
idx, score, method = find_game("xyzabc completely fake game")

print("Test 3")
print("Index:", idx)
print("Score:", score)
print("Method:", method)

Test 3
Index: None
Score: 0
Method: none


In [63]:
# Step 3 Exit Check

test_queries = [
    "Elden Ring",
    "tell me about Elden Ring",
    "eldn ring",
    "witcher 3",
    "xyzabc completely fake game"
]

for query in test_queries:
    idx, score, method = find_game(query)

    game = df.loc[idx, "name"] if idx is not None else None

    print(f"Query: {query}")
    print(f"Game: {game}")
    print(f"Score: {score}")
    print(f"Method: {method}")
    print("-" * 50)

Query: Elden Ring
Game: ELDEN RING
Score: 100
Method: exact
--------------------------------------------------
Query: tell me about Elden Ring
Game: ELDEN RING
Score: 95
Method: substring
--------------------------------------------------
Query: eldn ring
Game: ELDEN RING
Score: 94.73684210526316
Method: fuzzy
--------------------------------------------------
Query: witcher 3
Game: The Witcher 3: Wild Hunt
Score: 95
Method: substring
--------------------------------------------------
Query: xyzabc completely fake game
Game: None
Score: 0
Method: none
--------------------------------------------------


In [64]:
idx, score, method = find_game("witcher 3")

print("Index:", idx)
print("Score:", score)
print("Method:", method)

if idx is not None:
    print("Game:", df.loc[idx, "name"])

Index: 15
Score: 95
Method: substring
Game: The Witcher 3: Wild Hunt


# Step 3 — Game Name Index & Lookup Complete

The GameGuide game-name lookup system has been successfully implemented and tested.

### Completed Tasks

- Created `name_clean` with Unicode-aware normalization.
- Removed 4 unusable symbol-only game names.
- Built `name_index` mapping normalized game names to DataFrame indices.
- Resolved duplicate normalized names by prioritizing games with the highest review count.
- Implemented three matching layers:
  1. Exact matching
  2. Substring matching
  3. Fuzzy matching using RapidFuzz
- Added top-3 fuzzy game suggestions using `suggest()`.
- Added safeguards against short-name and fuzzy-match false positives.
- Validated common game queries, misspellings, partial names, and invalid queries.

### Final Lookup Behavior

| Query Type | Example | Result |
|---|---|---|
| Exact | `Elden Ring` | ELDEN RING |
| Substring | `tell me about Elden Ring` | ELDEN RING |
| Partial | `witcher 3` | The Witcher 3: Wild Hunt |
| Typo | `eldn ring` | ELDEN RING |
| Invalid | `xyzabc completely fake game` | No match |

### Step 3 Exit Check

All required lookup tests passed successfully.

The GameGuide name index is now ready to support entity extraction, information retrieval, search, and recommendation modules.